:important: This is the notebook for my Wildfires project for SDS210. Broad structure as follows:

1. Import required packages
    

2. test pull data into project with the FIRMS API and then just use the bounding box for Australia.

3. reproject to correct CRS


4. 

In [1]:
import requests
import pandas as pd
import geopandas as gpd
import time
import datetime
import folium
from folium.plugins import MarkerCluster
import numpy as np

In [2]:
# We need to access the API and to do that, will use the map key that permits access.
MAP_KEY = '54684dde74a099b139ddbbef0f621891'

# Now let's check how many results we have

url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  test_response = requests.get(url)
  test_data = response.json()
  test_df = pd.Series(data)
  display(test_df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

There is an issue with the query. 
Try in your browser: https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=54684dde74a099b139ddbbef0f621891


In [3]:
# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
sensor_data = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
test_df = pd.read_csv(sensor_data)
display(test_df)

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-18
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-18
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-18
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-18
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-18
8,GOES_NRT,2022-08-09,2026-05-18
9,BA_MODIS,2000-11-01,2026-02-01


In [4]:
# We are particularly interested in the wildfires in Australia and so will select this information using a bounding box. Aus = 110 -55, 180 -10
modis_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_NRT/110,-50,160,-11/3'
modis_nrt_df = pd.read_csv(modis_nrt_url)

modis_sp_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_SP/110,-50,160,-11/3'
modis_sp_df = pd.read_csv(modis_sp_url)

viirs_noaa20_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/110,-50,160,-11/3'
viirs_noaa20_nrt_df = pd.read_csv(viirs_noaa20_nrt_url)

viirs_noaa20_sp_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_SP/110,-50,160,-11/3'
viirs_noaa20_sp_df = pd.read_csv(viirs_noaa20_sp_url)

viirs_noaa21_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA21_NRT/110,-50,160,-11/3'
viirs_noaa21_nrt_df = pd.read_csv(viirs_noaa21_nrt_url)

viirs_snpp_nrt_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/110,-50,160,-11/3'
viirs_snpp_nrt_df = pd.read_csv(viirs_snpp_nrt_url)

viirs_snpp_sp_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_SP/110,-50,160,-11/3'
viirs_snpp_sp_df = pd.read_csv(viirs_snpp_sp_url)

In [5]:
print(modis_nrt_df.iloc[100,:])

latitude       -15.18039
longitude      127.64879
brightness        319.02
scan                1.17
track               1.08
acq_date      2026-05-16
acq_time             652
satellite           Aqua
instrument         MODIS
confidence            71
version           6.1NRT
bright_t31        299.17
frp                 11.2
daynight               D
Name: 100, dtype: object


In [6]:
# Create a funtion that checks time of aquisition and calculates time delta
def add_time_since_acq(df):
    df["acq_date"] = pd.to_datetime(df["acq_date"])   # Ensuring date is in datetime format
    df["acq_datetime"] = pd.to_datetime(
        df["acq_date"].astype(str) + df["acq_time"].astype(str).str.zfill(4),    # Turns all values into 4 digit HHHH format
        format = "%Y-%m-%d%H%M"
        ).dt.tz_localize('UTC')

    current_time = pd.Timestamp.now('UTC')
    df["time_since_detection"] = current_time - df["acq_datetime"]

    # Calculate hours since detection
    df["hours_since"] = df["time_since_detection"].dt.total_seconds()/3600

    return df


modis_nrt_df = add_time_since_acq(modis_nrt_df)
print(modis_nrt_df.iloc[100,:])

latitude                                -15.18039
longitude                               127.64879
brightness                                 319.02
scan                                         1.17
track                                        1.08
acq_date                      2026-05-16 00:00:00
acq_time                                      652
satellite                                    Aqua
instrument                                  MODIS
confidence                                     71
version                                    6.1NRT
bright_t31                                 299.17
frp                                          11.2
daynight                                        D
acq_datetime            2026-05-16 06:52:00+00:00
time_since_detection       2 days 14:02:59.002012
hours_since                             62.049723
Name: 100, dtype: object


In [7]:
# Convert to GeoDataFrame (WGS84)
modis_nrt_gdf = gpd.GeoDataFrame(
    modis_nrt_df, 
    geometry=gpd.points_from_xy(
        modis_nrt_df["longitude"], 
        modis_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

modis_sp_gdf = gpd.GeoDataFrame(
    modis_sp_df, 
    geometry=gpd.points_from_xy(
        modis_sp_df["longitude"], 
        modis_sp_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_noaa20_nrt_gdf = gpd.GeoDataFrame(
    viirs_noaa20_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_noaa20_nrt_df["longitude"], 
        viirs_noaa20_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_noaa20_sp_gdf = gpd.GeoDataFrame(
    viirs_noaa20_sp_df, 
    geometry=gpd.points_from_xy(
        viirs_noaa20_sp_df["longitude"], 
        viirs_noaa20_sp_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_noaa21_nrt_gdf = gpd.GeoDataFrame(
    viirs_noaa21_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_noaa21_nrt_df["longitude"], 
        viirs_noaa21_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_snpp_nrt_gdf = gpd.GeoDataFrame(
    viirs_snpp_nrt_df, 
    geometry=gpd.points_from_xy(
        viirs_snpp_nrt_df["longitude"], 
        viirs_snpp_nrt_df["latitude"]
    ),
    crs="EPSG:4326")

viirs_snpp_sp_gdf = gpd.GeoDataFrame(
    viirs_snpp_sp_df, 
    geometry=gpd.points_from_xy(
        viirs_snpp_sp_df["longitude"], 
        viirs_snpp_sp_df["latitude"]
    ),
    crs="EPSG:4326")


print(f"Found {len(modis_nrt_gdf)} fire records.")
print(f"Found {len(modis_sp_gdf)} fire records.")
print(f"Found {len(viirs_noaa20_nrt_gdf)} fire records.")
print(f"Found {len(viirs_noaa20_sp_gdf)} fire records.")
print(f"Found {len(viirs_noaa21_nrt_gdf)} fire records.")
print(f"Found {len(viirs_snpp_nrt_gdf)} fire records.")
print(f"Found {len(viirs_snpp_sp_gdf)} fire records.")

Found 677 fire records.
Found 0 fire records.
Found 3912 fire records.
Found 0 fire records.
Found 3237 fire records.
Found 3517 fire records.
Found 0 fire records.


In [8]:
# Initialise a map centered on Australia
australia_map = folium.Map(
    location=[-28.281828, 136.145401],
    zoom_start=5,
    tiles="CyclOSM",  # A clean, light basemap
)

modis_nrt_gdf.explore(
    m=australia_map, 
    column='confidence', # This is what the scale is based on.
    name='MODIS NRT',  
    tooltip=['brightness', 'confidence'], 
    cmap='YlOrRd',
    style_kwds={'fillOpacity': 0.8, 'color': 'white', 'weight': 0.1},
    show=True
)

viirs_noaa20_nrt_gdf.explore(
    m=australia_map, 
    column='confidence', # This is what the scale is based on.
    name='VIIRS NOAA20 NRT',  
    tooltip=['bright_ti4', 'confidence'], 
    cmap='YlOrRd',
    style_kwds={'fillOpacity': 0.8, 'color': 'white', 'weight': 0.1},
    show=True
)

folium.LayerControl().add_to(australia_map)

australia_map

TypeError: Object of type Timedelta is not JSON serializable

In [9]:
# Initialise a map centered on Australia
australia_map = folium.Map(
    location=[-28.281828, 136.145401],
    zoom_start=5,
    tiles="CyclOSM",  # A clean, light basemap
)

# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(
    modis_nrt_gdf,
    name = "MODIS_NRT",
    tooltip=folium.GeoJsonTooltip(fields=["confidence"], aliases=["Confidence (%)"]),
    marker=folium.CircleMarker(
        radius=4,           # size in pixels
        color="black",      # border color
        weight=1,           # border thickness
        fill=True,
        fill_color="orange",
        fill_opacity=0.7,
    ),
).add_to(australia_map)

# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(
    modis_sp_gdf,
    name = "MODIS_SP",
    #tooltip=folium.GeoJsonTooltip(fields=[""], aliases=[""]),
    marker=folium.CircleMarker(
        radius=4,           # size in pixels
        color="blue",      # border color
        weight=1,            # border thickness
        fill=True,
        fill_color="orange",
        fill_opacity=0.7,
    ),
).add_to(australia_map)

# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(
    viirs_noaa20_nrt_gdf,
    name = "VIIRS NOAA20 NRT",
    #tooltip=folium.GeoJsonTooltip(fields=[""], aliases=[""]),
    marker=folium.CircleMarker(
        radius=4,           # size in pixels
        color="blue",      # border color
        weight=1,            # border thickness
        fill=True,
        fill_color="orange",
        fill_opacity=0.7,
    ),
).add_to(australia_map)


# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(
    viirs_noaa20_sp_gdf,
    name = "VIIRS NOAA20 SP",
    #tooltip=folium.GeoJsonTooltip(fields=[""], aliases=[""]),
    marker=folium.CircleMarker(
        radius=4,           # size in pixels
        color="black",      # border color
        weight=1,           # border thickness
        fill=True,
        fill_color="orange",
        fill_opacity=0.7,
    ),
).add_to(australia_map)


# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(
    viirs_noaa21_nrt_gdf,
    name = "VIIRS NOAA21 NRT",
    #tooltip=folium.GeoJsonTooltip(fields=[""], aliases=[""]),
    marker=folium.CircleMarker(
        radius=4,           # size in pixels
        color="black",      # border color
        weight=1,           # border thickness
        fill=True,
        fill_color="orange",
        fill_opacity=0.7,
    ),
).add_to(australia_map)


# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(
    viirs_snpp_nrt_gdf,
    name = "VIIRS SNPP NRT",
   #tooltip=folium.GeoJsonTooltip(fields=[""], aliases=[""]),
    marker=folium.CircleMarker(
        radius=4,           # size in pixels
        color="black",      # border color
        weight=1,           # border thickness
        fill=True,
        fill_color="orange",
        fill_opacity=0.7,
    ),
).add_to(australia_map)


# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(
    viirs_snpp_sp_gdf,
    name = "VIIRS SNPP SP",
    #tooltip=folium.GeoJsonTooltip(fields=[""], aliases=[""]),
    marker=folium.CircleMarker(
        radius=4,           # size in pixels
        color="black",      # border color
        weight=1,           # border thickness
        fill=True,
        fill_color="orange",
        fill_opacity=0.7,
    ),
).add_to(australia_map)

# Add interactive layer control menu to the top right corner
folium.LayerControl().add_to(australia_map)


australia_map1 = australia_map.explore()


TypeError: Object of type Timestamp is not JSON serializable

In [ ]:
# 1. Prepare the data
quarters = gpd.read_file("data/LGA_2025_AUST_GDA2020")

quarters.head

In [ ]:
# 1. Prepare the data
quarters = gpd.read_file("data/LGA_2025_AUST_GDA2020")

# Calculate the area in square kilometers
quarters["area"] = (quarters.geometry.area / 1000000).round(3)

# 2. Initialize the Folium Map
m = folium.Map(
    location=[47.3769, 8.5417], zoom_start=12, tiles="CartoDB Positron No Labels"
)

# 3. Add the Choropleth layer
folium.Choropleth(
    geo_data=quarters,  # The spatial geometries
    name="Quarter Area",  # Name for the LayerControl
    data=quarters,  # The tabular data source containing the values
    columns=["name", "area"],  # [The Key column to match, The Value column to color by]
    key_on="feature.properties.name",  # The exact path to the key inside the GeoJSON structure
    fill_color="viridis",  # The color palette
    fill_opacity=0.6,  # Transparency of the polygons
    line_opacity=0.2,  # Transparency of the borders
    legend_name="Area in km2",  # Title for the automatically generated legend
).add_to(m)

# 4. Add layer control to toggle the map on and off
folium.LayerControl().add_to(m)

m

In [ ]:
import os
print(os.getcwd())
print(os.listdir())